# Pull every trade for one closed Polymarket market

1. Resolve a closed market by slug (Gamma API).
2. Page through all of its trades (Data API `/v2/trades`).
3. Load the trades into a DataFrame and show a few rows.

Uses the repo's read-only `polyresearch.client`, so no credentials are needed.

In [35]:
import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

# Make the repo root importable when the kernel starts in notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from polyresearch.client import Client, DATA, GAMMA

client = Client()

In [36]:
# A closed finance market: S&P 500 (SPX) Up or Down on September 11, 2026 (resolved Up)
SLUG = "spx-up-or-down-on-september-11-2026"

# Gamma hides closed markets from slug lookups unless closed=True is passed
(market,) = client.get(GAMMA, "/markets", slug=SLUG, closed=True)
assert market["closed"], "expected a closed market"

pd.Series({
    "question": market["question"],
    "condition_id": market["conditionId"],
    "closed": market["closed"],
    "end_date": market["endDate"],
    "outcomes": market["outcomes"],
    "final_prices": market["outcomePrices"],
    "volume_usdc": round(market["volumeNum"], 2),
})

question                S&P 500 (SPX) Up or Down on September 11?
condition_id    0x7d8d0aa3c8445853a8d7910268068cd006cd42930416...
closed                                                       True
end_date                                     2026-09-11T20:00:00Z
outcomes                                           ["Up", "Down"]
final_prices                                           ["1", "0"]
volume_usdc                                             147780.17
dtype: object

In [37]:
PAGE_SIZE = 1000      # API maximum per request
PAUSE_SECONDS = 1.0   # wait between requests so we stay well under the rate limit
MAX_PAGES = 500       # safety cap: 500k rows

def fetch_all_trades(taker_only=True, max_pages=MAX_PAGES, pause=PAUSE_SECONDS,
                     deadline=None, api=None, verbose=True, **filters):
    """Walk /v2/trades cursor by cursor until the feed says there is nothing more.

    filters: condition=<condition id> for a market, or user=<wallet> (+ start/end) for a wallet.
    deadline: a time.monotonic() value; stop early once it passes.
    Returns (rows, status), status is "complete", "page_cap" or "deadline".
    The client also backs off and retries on HTTP 429 (honouring Retry-After).
    """
    api = api or client
    params = {**filters, "taker_only": taker_only, "limit": PAGE_SIZE}
    rows, seen_cursors = [], set()
    for page_no in range(1, max_pages + 1):
        if deadline is not None and time.monotonic() >= deadline:
            return rows, "deadline"
        page = api.get(DATA, "/v2/trades", **params)
        rows.extend(page["data"])
        if verbose:
            print(f"  page {page_no}: +{len(page['data'])} rows ({len(rows):,} so far)")
        pagination = page["pagination"]
        if not pagination["has_more"]:
            return rows, "complete"
        cursor = pagination["next_cursor"]
        if not cursor or cursor in seen_cursors:
            raise RuntimeError("API returned a missing or repeated cursor")
        seen_cursors.add(cursor)
        params["cursor"] = cursor
        time.sleep(pause)
    return rows, "page_cap"

# One row per trade: the taker side of each fill
print("Taker fills (one row per trade):")
rows, status = fetch_all_trades(condition=market["conditionId"], taker_only=True)
assert status == "complete", "hit MAX_PAGES before the end of the feed"
time.sleep(PAUSE_SECONDS)

# Every leg: the taker plus each maker order it matched against
print("All fill legs (taker + makers):")
leg_rows, status = fetch_all_trades(condition=market["conditionId"], taker_only=False)
assert status == "complete", "hit MAX_PAGES before the end of the feed"
all_legs = pd.DataFrame(leg_rows)

Taker fills (one row per trade):
  page 1: +1000 rows (1,000 so far)
  page 2: +32 rows (1,032 so far)
All fill legs (taker + makers):
  page 1: +1000 rows (1,000 so far)
  page 2: +1000 rows (2,000 so far)
  page 3: +405 rows (2,405 so far)


In [38]:
trades = pd.DataFrame(rows)
trades["time_utc"] = pd.to_datetime(trades["timestamp"], unit="s", utc=True)
trades["notional_usdc"] = trades["size"] * trades["price"]
trades = trades.sort_values("time_utc").reset_index(drop=True)

cols = ["time_utc", "proxy_wallet", "pseudonym", "side", "outcome",
        "size", "price", "notional_usdc", "transaction_hash"]
trades[cols].head(10)

,time_utc,proxy_wallet,pseudonym,side,outcome,size,price,notional_usdc,transaction_hash
0,2026-09-10 19:59:04+00:00,0x28936c250fa8337e7d7c1adbfe9346eacaba4c6f,Simplistic-Personal,BUY,Up,6.000000,0.620000,3.7200,0x02ae819ed3081dbb164b7ebcd8443f42b3c30f69908a...
1,2026-09-10 20:02:16+00:00,0xdd1ee9e42f3bc2f5564de4efc81c8a2c3fc3ccc9,Indolent-Folder,BUY,Up,5.000000,0.600000,3.0000,0x1955cd4cd29b42c63ba069f5894362d268f50d6826af...
2,2026-09-10 20:13:00+00:00,0x6f2a0d4ff16a3ef378a65c62d20cde974578de7d,Petty-Archaeology,BUY,Down,43.990000,0.430000,18.9157,0xce408a5d610c5d6f1531e2a82279abb9570772a82e34...
3,2026-09-10 20:13:21+00:00,0x28936c250fa8337e7d7c1adbfe9346eacaba4c6f,Simplistic-Personal,BUY,Up,11.375000,0.560000,6.3700,0xbd203b58ba390ffdb364a61bff30db66ec76a23c03f7...
4,2026-09-10 20:14:16+00:00,0xdd1ee9e42f3bc2f5564de4efc81c8a2c3fc3ccc9,Indolent-Folder,BUY,Up,2.000000,0.560000,1.1200,0xa6527662165913d3ab42b6046ca66d924d55279c97fc...
5,2026-09-10 20:14:16+00:00,0xdd1ee9e42f3bc2f5564de4efc81c8a2c3fc3ccc9,Indolent-Folder,BUY,Up,2.000000,0.560000,1.1200,0x087d0bbccb80da0ef827d510d178f68566a9a41ba12c...
6,2026-09-10 20:18:01+00:00,0x1adbfeab19fe223e5244df8f6906c162d3b62940,Frilly-Guacamole,BUY,Down,132.660000,0.470000,62.3502,0xc59b3506534e4abb328f4ea3f9fd12ac800d70dfca54...
7,2026-09-10 20:18:49+00:00,0xb82530e5f6fb291ffd3df278561ae517ef2c0666,Jovial-Pathogenesis,SELL,Up,20.000000,0.520000,10.4000,0x2a12ddb8b2f49c2ed19289f410b50cab01f5fe9eaf7b...
8,2026-09-10 20:19:37+00:00,0x61234f998ff1a26aafe36125a4c204d30d9c8b33,,BUY,Up,18.078036,0.553157,10.0000,0xc1b7a9c9e79286ad8db1f5bd2852b8e265a68dccfc6b...
9,2026-09-10 20:46:48+00:00,0xeadf5963682dd237184f484da138ce74bfec26ef,Splendid-Convection,BUY,Up,130.580000,0.564586,73.7237,0x006b84c66051d36fc21094035ea486304a95a5d03ac2...


In [39]:
# Quick sanity summary
print(f"total trades on this market: {len(trades):,}")
print(f"total fill legs (taker + makers): {len(all_legs):,}")
print("first trade:", trades["time_utc"].min())
print("last trade: ", trades["time_utc"].max())
print("unique wallets:", trades["proxy_wallet"].nunique())
trades.groupby(["outcome", "side"])["notional_usdc"].agg(["count", "sum"]).round(2)

total trades on this market: 1,032
total fill legs (taker + makers): 2,405
first trade: 2026-09-10 19:59:04+00:00
last trade:  2026-09-11 20:41:15+00:00
unique wallets: 268


count       sum
outcome side                 
Down    BUY     530  21493.86
        SELL     72    664.39
Up      BUY     411  39414.32
        SELL     19   1974.87

## Split the trades into 5 equal time windows, open → close

`/v2/trades` **ignores `start`/`end` when filtering by `condition`** (it returns the full history no matter what), so we pull every trade once (above) and bucket them locally.

- **Open** = `acceptingOrdersTimestamp`: when the order book went live.
- **Close** = `closedTime`: when the market actually stopped trading. `endDate` is *not* the close; it is the scheduled end (here the 4pm ET market close), and trading continues until the market is resolved.

In [40]:
import numpy as np

N_INTERVALS = 5

opened = pd.to_datetime(market["acceptingOrdersTimestamp"], utc=True)
closed = pd.to_datetime(market["closedTime"], utc=True)

# N+1 evenly spaced edges -> N equal-width windows
edges = pd.to_datetime(
    np.linspace(opened.value, closed.value, N_INTERVALS + 1).astype("int64"), utc=True
)
print(f"open:  {opened}")
print(f"close: {closed}")
print(f"window width: {(closed - opened) / N_INTERVALS}")

outside = trades[(trades["time_utc"] < opened) | (trades["time_utc"] > closed)]
print(f"trades outside open→close: {len(outside)}")

trades["interval"] = pd.cut(
    trades["time_utc"], bins=edges, labels=range(1, N_INTERVALS + 1), include_lowest=True
)

open:  2026-09-10 12:00:56+00:00
close: 2026-09-11 22:28:22+00:00
window width: 0 days 06:53:29.200000
trades outside open→close: 0


In [41]:
# Express every fill as the implied probability of the FIRST outcome,
# so both sides sit on one scale (a Down fill at 0.40 means Up at 0.60).
first_outcome = json.loads(market["outcomes"])[0]
trades["p_first"] = np.where(
    trades["outcome"] == first_outcome, trades["price"], 1 - trades["price"]
)

def summarize(g):
    return pd.Series({
        "trades": len(g),
        "wallets": g["proxy_wallet"].nunique(),
        "notional_usdc": g["notional_usdc"].sum(),
        f"vwap_p_{first_outcome}": np.average(g["p_first"], weights=g["notional_usdc"]) if len(g) else np.nan,
        f"last_p_{first_outcome}": g["p_first"].iloc[-1] if len(g) else np.nan,
    })

summary = trades.groupby("interval", observed=False).apply(summarize, include_groups=False)
summary.insert(0, "start", edges[:-1])
summary.insert(1, "end", edges[1:])
summary = summary.astype({"trades": int, "wallets": int})
summary.round({"notional_usdc": 2, f"vwap_p_{first_outcome}": 3, f"last_p_{first_outcome}": 3})

,start,end,trades,wallets,notional_usdc,vwap_p_Up,last_p_Up
interval,,,,,,,
1,2026-09-10 12:00:56.000+00:00,2026-09-10 18:54:25.200+00:00,0,0,0.00,NaN,NaN
2,2026-09-10 18:54:25.200+00:00,2026-09-11 01:47:54.400+00:00,214,35,21992.43,0.658,0.660
3,2026-09-11 01:47:54.400+00:00,2026-09-11 08:41:23.600+00:00,320,141,11086.89,0.644,0.760
4,2026-09-11 08:41:23.600+00:00,2026-09-11 15:34:52.800+00:00,438,114,25822.00,0.828,0.990
5,2026-09-11 15:34:52.800+00:00,2026-09-11 22:28:22.000+00:00,60,21,4646.12,0.990,0.999


In [42]:
# One DataFrame per window, then a peek at the first few rows of each
intervals = {i: g[cols + ["p_first"]] for i, g in trades.groupby("interval", observed=False)}

for i, df in intervals.items():
    print(f"Interval {i}: {edges[i-1]:%Y-%m-%d %H:%M} → {edges[i]:%Y-%m-%d %H:%M} UTC, {len(df)} trades")
    display(df.head(3))

Interval 1: 2026-09-10 12:00 → 2026-09-10 18:54 UTC, 0 trades


,time_utc,proxy_wallet,pseudonym,side,outcome,size,price,notional_usdc,transaction_hash,p_first


Interval 2: 2026-09-10 18:54 → 2026-09-11 01:47 UTC, 214 trades


,time_utc,proxy_wallet,pseudonym,side,outcome,size,price,notional_usdc,transaction_hash,p_first
0,2026-09-10 19:59:04+00:00,0x28936c250fa8337e7d7c1adbfe9346eacaba4c6f,Simplistic-Personal,BUY,Up,6.00,0.62,3.7200,0x02ae819ed3081dbb164b7ebcd8443f42b3c30f69908a...,0.62
1,2026-09-10 20:02:16+00:00,0xdd1ee9e42f3bc2f5564de4efc81c8a2c3fc3ccc9,Indolent-Folder,BUY,Up,5.00,0.60,3.0000,0x1955cd4cd29b42c63ba069f5894362d268f50d6826af...,0.60
2,2026-09-10 20:13:00+00:00,0x6f2a0d4ff16a3ef378a65c62d20cde974578de7d,Petty-Archaeology,BUY,Down,43.99,0.43,18.9157,0xce408a5d610c5d6f1531e2a82279abb9570772a82e34...,0.57


Interval 3: 2026-09-11 01:47 → 2026-09-11 08:41 UTC, 320 trades


,time_utc,proxy_wallet,pseudonym,side,outcome,size,price,notional_usdc,transaction_hash,p_first
214,2026-09-11 01:48:13+00:00,0x56b0233010d8f8a5b0c53d70b59b62647e7e18e6,Spherical-Travel-Commuter,BUY,Down,250.0,0.34,85.0,0xe59c8091e567b032b3711cbf12c091f963c07088e34f...,0.66
215,2026-09-11 01:48:18+00:00,0x56b0233010d8f8a5b0c53d70b59b62647e7e18e6,Spherical-Travel-Commuter,BUY,Down,250.0,0.34,85.0,0xc99fe41a5383b71437bb62dcdad6386def336ee44cea...,0.66
216,2026-09-11 01:48:28+00:00,0x56b0233010d8f8a5b0c53d70b59b62647e7e18e6,Spherical-Travel-Commuter,BUY,Down,250.0,0.34,85.0,0xfed68b048d1d99f686fa51f96797a6650db7623ad9ff...,0.66


Interval 4: 2026-09-11 08:41 → 2026-09-11 15:34 UTC, 438 trades


,time_utc,proxy_wallet,pseudonym,side,outcome,size,price,notional_usdc,transaction_hash,p_first
534,2026-09-11 08:47:37+00:00,0x982a77e75498fa134607f598c861b8de9d0c6851,Snarling-Pantology,BUY,Down,9.990000,0.24,2.397600,0xa1e69805547530fbc99bc0bb49a426277217ba862a7a...,0.76
535,2026-09-11 08:53:13+00:00,0x9e3ed7b661a903fc97afcf49e0f014ebe869f882,Favorable-Pepper-Brushing,BUY,Up,108.402595,0.77,83.469998,0xc45ffc1745fbbca641752280d6854f045c559dd84648...,0.77
536,2026-09-11 08:54:01+00:00,0xf60ec79ce74057d448754b9f51195d13b085936d,Old-Fashioned-Ladle,BUY,Up,26.000000,0.78,20.280000,0xf6f6b4ccb4615aa79e7078e724c1326550f689b62676...,0.78


Interval 5: 2026-09-11 15:34 → 2026-09-11 22:28 UTC, 60 trades


,time_utc,proxy_wallet,pseudonym,side,outcome,size,price,notional_usdc,transaction_hash,p_first
972,2026-09-11 15:54:21+00:00,0x06ce4f6a73871f2026003b611ba4ca9ab8c70b8b,Frail-Item,SELL,Down,15.46,0.01000,0.1546,0xab055404e134f1898e25d0b257113d1634051d7fe322...,0.99000
973,2026-09-11 16:06:36+00:00,0x50c80ae7a66e49701dc9388bd13574ba03e0caa2,Clean-Derivative,BUY,Down,200.00,0.01525,3.0500,0xa668f531e94051e1405382182e69c10e0d57b871e29a...,0.98475
974,2026-09-11 16:53:19+00:00,0x50c80ae7a66e49701dc9388bd13574ba03e0caa2,Clean-Derivative,BUY,Down,200.00,0.01200,2.4000,0xecc5ac4ea80ae914de0f301dbc3f3589f82f8095b159...,0.98800


## How many other markets had each wallet traded before this one?

For every wallet on this market (takers **and** makers), pull its trade history from `/v2/trades?user=…` (both roles; user requests do honour `end`), then count distinct *other* markets (`condition_id`s):

- **`prior_markets`**: other markets traded **before the wallet's first trade here**. Point-in-time, no look-ahead.
- **`other_markets_by_close`**: other markets traded any time up to this market's close.

Histories are only fetched up to the close, so nothing after the market resolved leaks in.

**Time-boxed:** the walk stops after `WALLET_TIME_BUDGET` seconds (default 240s, so the cell finishes in under ~5 min). Wallets are visited biggest-volume-on-this-market first, so a cut-off loses the least important ones. Wallets with more than `WALLET_MAX_PAGES × 1000` fills are flagged `history_truncated`; their counts are lower bounds. **Re-run the cell to continue** from where it stopped.

In [43]:
WALLET_TIME_BUDGET = 240   # seconds for the whole walk; hard stop
WALLET_MAX_PAGES = 5       # per-wallet cap (5k fills); heavier wallets are flagged as truncated
WALLET_PAUSE = 0.3         # seconds between wallet requests (≈3 req/s, gentle on the API)
wallet_client = Client(timeout=10, retries=1)  # short timeouts so one slow request can't eat the budget

close_ts = int(closed.timestamp())
# Biggest wallets on this market first, so a cut-off drops the least important ones
wallets = (all_legs.assign(wallet=all_legs["proxy_wallet"].str.lower())
           .groupby("wallet").apply(lambda g: (g["size"] * g["price"]).sum(), include_groups=False)
           .sort_values(ascending=False).index.tolist())
print(f"{len(wallets)} wallets traded on this market (takers and makers)")

# Kept across re-runs so an interrupted walk resumes instead of starting over
if "wallet_history" not in globals():
    wallet_history, wallet_truncated, wallet_errors = {}, {}, {}

started = time.monotonic()
deadline = started + WALLET_TIME_BUDGET
for n, wallet in enumerate(wallets, 1):
    if wallet in wallet_history:
        continue
    if time.monotonic() >= deadline:
        break
    try:
        hist, status = fetch_all_trades(
            user=wallet, taker_only=False, end=close_ts, api=wallet_client,
            max_pages=WALLET_MAX_PAGES, pause=WALLET_PAUSE, deadline=deadline, verbose=False,
        )
    except Exception as error:  # keep going; failed wallets are listed below
        wallet_errors[wallet] = str(error)
        continue
    if status == "deadline":     # partial history: discard it, refetch on the next run
        break
    wallet_errors.pop(wallet, None)
    wallet_history[wallet], wallet_truncated[wallet] = hist, status == "page_cap"
    if n % 10 == 0:
        print(f"  {n}/{len(wallets)} wallets, {time.monotonic() - started:.0f}s elapsed")
    time.sleep(WALLET_PAUSE)

remaining = [w for w in wallets if w not in wallet_history]
print(f"done in {time.monotonic() - started:.0f}s — fetched: {len(wallet_history)}/{len(wallets)}, "
      f"truncated: {sum(wallet_truncated.values())}, failed: {len(wallet_errors)}")
if remaining:
    print(f"{len(remaining)} wallets not fetched yet; re-run this cell to continue")

329 wallets traded on this market (takers and makers)
  10/329 wallets, 32s elapsed
  20/329 wallets, 67s elapsed
  30/329 wallets, 96s elapsed
  40/329 wallets, 127s elapsed
  50/329 wallets, 150s elapsed
  60/329 wallets, 175s elapsed
  70/329 wallets, 202s elapsed
  80/329 wallets, 230s elapsed
done in 240s — fetched: 83/329, truncated: 42, failed: 0
246 wallets not fetched yet; re-run this cell to continue


In [45]:
cid = market["conditionId"]
legs = all_legs.assign(wallet=all_legs["proxy_wallet"].str.lower())
first_here = legs.groupby("wallet")["timestamp"].min()
pseudonyms = legs.groupby("wallet")["pseudonym"].first()

records = []
for wallet, hist in wallet_history.items():
    h = pd.DataFrame(hist, columns=["condition_id", "timestamp"])
    h = h[h["timestamp"] < close_ts]          # enforce the end filter locally too
    other = h[h["condition_id"] != cid]
    records.append({
        "wallet": wallet,
        "pseudonym": pseudonyms[wallet],
        "first_trade_here": pd.to_datetime(first_here[wallet], unit="s", utc=True),
        "prior_markets": other.loc[other["timestamp"] < first_here[wallet], "condition_id"].nunique(),
        "other_markets_by_close": other["condition_id"].nunique(),
        "history_fills": len(h),
        "history_truncated": wallet_truncated[wallet],
    })

wallet_markets = (pd.DataFrame(records).set_index("wallet")
                  .sort_values("prior_markets", ascending=False))

print(f"wallets with no prior markets (first-timers): {(wallet_markets['prior_markets'] == 0).sum()} of {len(wallet_markets)}")
display(wallet_markets["prior_markets"].describe().round(1).to_frame().T)
wallet_markets.head(15)

wallets with no prior markets (first-timers): 4 of 83


,count,mean,std,min,25%,50%,75%,max
prior_markets,83.0,634.1,822.1,0.0,87.5,393.0,770.5,4156.0


,pseudonym,first_trade_here,prior_markets,other_markets_by_close,history_fills,history_truncated
wallet,,,,,,
0x8ceb9a444c91f460973f7ccd6adf1dcd56aa1952,Rotating-Pith,2026-09-11 18:00:27+00:00,4156,4156,5000,True
0xfb8721d042d2d84b07e5735b0881d1bacc1bf53d,Gummy-Hook,2026-09-11 07:36:58+00:00,4151,4157,5000,True
0x1adbfeab19fe223e5244df8f6906c162d3b62940,Frilly-Guacamole,2026-09-10 20:18:01+00:00,3284,3312,5000,True
0xe6e695c5734405cdb28dd49c7674be4e592679b5,Harmful-Chairperson,2026-09-11 13:40:46+00:00,2607,2608,5000,True
0xc4ca43d6909192021cac77bea7b0ad07acc23df3,,2026-09-11 20:03:07+00:00,1817,1817,4256,False
0xe3164027a2be859579fcd84fce34513f0bc9bcf9,Likely-Submitter,2026-09-11 16:53:19+00:00,1800,1805,3486,False
0x4025a99c01cbf42511f850865d76d0eba75088b6,Tinted-Kind,2026-09-11 14:01:42+00:00,1736,1739,2769,False
0x46299616ff1eb6544ca3ba653d431a20f207f3b1,,2026-09-11 04:00:51+00:00,1670,1710,5000,True
0x62365f754d4aa7dc537f997f23bb5dd9a9e36881,Wet-Event,2026-09-10 20:13:00+00:00,1578,1621,5000,True


## Rank the 20 biggest wallets: skilled vs unskilled

**Point-in-time, no look-ahead.** For each wallet we take its trade history **before its first trade on this market**, and score only fills in markets that had **already resolved** by then. That's what you could have known about the wallet at the moment it showed up here.

**Scoring a fill against how its market resolved** (payout = 1 if the outcome won, else 0):

| | P&L | Capital at risk (stake) |
|---|---|---|
| BUY `size` @ `price` | `size × (payout − price)` | `size × price` |
| SELL `size` @ `price` | `size × (price − payout)` | `size × (1 − price)` |

Summed over a market, this equals the wallet's realised P&L there (whether it closed by trading or held to redemption).

**Per wallet:** total P&L, ROI (= P&L ÷ stake), win rate across markets, and a **t-statistic on ROI**: ROI divided by its stake-weighted standard error across markets. The t-stat is the ranking key. It follows the money (a wallet that wins many \$1 bets but loses its \$10k ones scores negative), and it rewards a *consistent* edge over a few lucky big wins. **`effective_markets`** is how many equal-sized bets the history is worth: 300 markets where one bet dominates count as ≈1–2.

| label | rule |
|---|---|
| **skilled** | ≥ `MIN_MARKETS` effective markets and t ≥ 2 |
| **unskilled** | ≥ `MIN_MARKETS` effective markets and t ≤ −2 |
| **no clear edge** | enough markets, but \|t\| < 2 |
| **insufficient history** | fewer than `MIN_MARKETS` effective markets |

**Caveats:** fees, splits/merges and neg-risk conversions aren't in the trades feed, so P&L is approximate. Histories are capped at the most recent `SKILL_MAX_PAGES × 1000` fills. Wallets with `avg_buy_price` ≥ `SNIPER_BUY_PRICE` get `style = resolution sniper`: *resolution sniping* (buying near-certain outcomes for a cent), which is profitable but a different thing from forecasting skill. Also, 20 wallets is a small sample.

In [ ]:
SKILL_N_WALLETS = 20
SKILL_MAX_PAGES = 5     # most recent 5k fills before each wallet's first trade here
SKILL_PAUSE = 0.3       # seconds between requests
skill_client = Client(timeout=10, retries=2)

def username(name, wallet):
    """`name` holds "<address>-<number>" when the user never picked a username."""
    return None if not name or name.lower().startswith(wallet) else name

legs = all_legs.assign(wallet=all_legs["proxy_wallet"].str.lower(),
                       notional=all_legs["size"] * all_legs["price"])
top = (legs.groupby("wallet")
           .agg(volume_here=("notional", "sum"), first_trade_here=("timestamp", "min"),
                name=("name", "first"), pseudonym=("pseudonym", "first"))
           .sort_values("volume_here", ascending=False)
           .head(SKILL_N_WALLETS))
top["username"] = [username(n, w) for w, n in top["name"].items()]

# Kept across re-runs, so an interrupted fetch resumes
if "skill_history" not in globals():
    skill_history = {}

started = time.monotonic()
for n, (wallet, row) in enumerate(top.iterrows(), 1):
    if wallet in skill_history:
        continue
    hist, status = fetch_all_trades(
        user=wallet, taker_only=False, end=int(row["first_trade_here"]) - 1,  # `end` is inclusive
        api=skill_client, max_pages=SKILL_MAX_PAGES, pause=SKILL_PAUSE, verbose=False,
    )
    skill_history[wallet] = (hist, status)
    label = next((x for x in (row["username"], row["pseudonym"]) if isinstance(x, str) and x), wallet[:10])
    print(f"  {n:2}/{len(top)} {label}: {len(hist):,} prior fills{' (capped)' if status != 'complete' else ''}")
    time.sleep(SKILL_PAUSE)
print(f"histories fetched in {time.monotonic() - started:.0f}s")

In [ ]:
# Look up how every market in those histories resolved (Gamma takes ~50 condition ids per request)
RESOLVE_BATCH = 50

def parse_resolution(m):
    prices = [float(x) for x in json.loads(m.get("outcomePrices") or "[]")]
    closed_at = pd.to_datetime(m.get("closedTime"), utc=True, errors="coerce")
    resolved = (len(prices) >= 2 and all(x in (0.0, 1.0) for x in prices)
                and sum(prices) == 1 and pd.notna(closed_at))
    return {"payouts": prices if resolved else None,
            "closed_ts": int(closed_at.timestamp()) if resolved else None,
            "question": m.get("question")}

if "resolutions" not in globals():
    resolutions = {}

needed = sorted({r["condition_id"].lower() for hist, _ in skill_history.values() for r in hist}
                - resolutions.keys())
print(f"{len(needed):,} markets to look up ({len(resolutions):,} cached)")
started = time.monotonic()
for i in range(0, len(needed), RESOLVE_BATCH):
    batch = needed[i:i + RESOLVE_BATCH]
    for m in skill_client.get(GAMMA, "/markets", condition_ids=batch, closed=True, limit=RESOLVE_BATCH):
        resolutions[m["conditionId"].lower()] = parse_resolution(m)
    for c_id in batch:  # not returned = still open (or unknown): never counted
        resolutions.setdefault(c_id, {"payouts": None, "closed_ts": None, "question": None})
    time.sleep(SKILL_PAUSE)

resolved_count = sum(r["payouts"] is not None for r in resolutions.values())
print(f"looked up in {time.monotonic() - started:.0f}s — "
      f"{resolved_count:,} of {len(resolutions):,} have a clean 0/1 resolution")

In [ ]:
MIN_MARKETS = 10       # effective markets needed before we'll call a wallet skilled or unskilled
SNIPER_BUY_PRICE = 0.95  # avg buy price at/above this = mostly buying near-certain outcomes

# 1) Score every prior fill against how its market resolved,
#    using only markets that resolved before the wallet arrived here
scored = []
for wallet, (hist, _) in skill_history.items():
    as_of = int(top.loc[wallet, "first_trade_here"])
    for r in hist:
        res = resolutions.get(r["condition_id"].lower())
        if (res is None or res["payouts"] is None or res["closed_ts"] >= as_of
                or int(r["timestamp"]) >= as_of):
            continue
        payout = res["payouts"][int(r["outcome_index"])]
        size, price, buy = float(r["size"]), float(r["price"]), r["side"] == "BUY"
        scored.append({
            "wallet": wallet, "condition_id": r["condition_id"].lower(), "buy": buy,
            "price": price, "size": size,
            "pnl": size * ((payout - price) if buy else (price - payout)),
            "stake": size * (price if buy else 1 - price),
        })
scored = pd.DataFrame(scored, columns=["wallet", "condition_id", "buy", "price", "size", "pnl", "stake"])

# 2) Roll fills up to one row per (wallet, market)
per_market = (scored.groupby(["wallet", "condition_id"])
                    .agg(pnl=("pnl", "sum"), stake=("stake", "sum"), fills=("pnl", "size"))
                    .query("stake > 0"))
per_market["ret"] = per_market["pnl"] / per_market["stake"]

# 3) Per-wallet skill statistics
def wallet_stats(g):
    w, r = g["stake"], g["ret"]
    roi = g["pnl"].sum() / w.sum()                      # = stake-weighted mean of per-market returns
    se = np.sqrt((w**2 * (r - roi) ** 2).sum()) / w.sum()  # its standard error
    return pd.Series({
        "resolved_markets": len(g),
        # How many equal-sized bets this is "worth": 300 markets where one bet dominates ≈ 1-2
        "effective_markets": w.sum() ** 2 / (w**2).sum(),
        "fills": g["fills"].sum(),
        "pnl_usdc": g["pnl"].sum(),
        "roi": roi,
        "win_rate": (g["pnl"] > 0).mean(),
        "t_stat": roi / se if len(g) > 1 and se > 0 else np.nan,
    })

stats = per_market.groupby("wallet").apply(wallet_stats, include_groups=False)
buys = scored[scored["buy"]]
stats["avg_buy_price"] = ((buys["size"] * buys["price"]).groupby(buys["wallet"]).sum()
                          / buys.groupby("wallet")["size"].sum())

def classify(row):
    if not row["effective_markets"] >= MIN_MARKETS:
        return "insufficient history"
    if row["t_stat"] >= 2:
        return "skilled"
    if row["t_stat"] <= -2:
        return "unskilled"
    return "no clear edge"

# 4) Sanity column: which side did each wallet take on THIS market, and was it right?
#    This market is out-of-sample for the ranking, so it's a (tiny) test of it.
first_outcome = json.loads(market["outcomes"])[0]
winner_is_first = json.loads(market["outcomePrices"])[0] == "1"
toward_first = np.where((legs["outcome"] == first_outcome) == (legs["side"] == "BUY"), 1, -1)
net_first_here = legs.assign(flow=toward_first * legs["notional"]).groupby("wallet")["flow"].sum()

skill = top[["username", "pseudonym", "volume_here"]].join(stats, how="left")
skill["resolved_markets"] = skill["resolved_markets"].fillna(0).astype(int)
skill["label"] = skill.apply(classify, axis=1)
skill["style"] = np.where(skill["avg_buy_price"] >= SNIPER_BUY_PRICE, "resolution sniper", "")
skill[f"net_{first_outcome}_here_usdc"] = net_first_here.reindex(skill.index)
skill["right_here"] = (skill[f"net_{first_outcome}_here_usdc"] > 0) == winner_is_first

order = {"skilled": 0, "no clear edge": 1, "unskilled": 2, "insufficient history": 3}
skill = (skill.assign(_order=skill["label"].map(order))
              .sort_values(["_order", "t_stat"], ascending=[True, False])
              .drop(columns="_order"))

print(skill["label"].value_counts().to_string(), "\n")
print(skill.groupby("label")["right_here"].agg(["sum", "count"])
           .rename(columns={"sum": "right_on_this_market", "count": "wallets"}).to_string(), "\n")
skill.style.format({
    "volume_here": "${:,.0f}", "pnl_usdc": "${:,.0f}", f"net_{first_outcome}_here_usdc": "${:,.0f}",
    "roi": "{:+.1%}", "win_rate": "{:.0%}", "effective_markets": "{:,.1f}",
    "t_stat": "{:+.2f}", "avg_buy_price": "{:.2f}", "fills": "{:,.0f}",
}, na_rep="–")